# Meta MMS (VITS) — Maithili TTS Evaluation

This notebook generates speech for 20 phonetically balanced Maithili sentences using the **Meta MMS** (`facebook/mms-tts-mai`) model,
then evaluates intelligibility via Whisper ASR (WER & CER).

**Model:** [facebook/mms-tts-mai](https://huggingface.co/facebook/mms-tts-mai)
**Language:** Maithili (mai)
**Architecture:** VITS-based (Meta MMS)


In [ ]:
!pip install torch transformers accelerate soundfile scipy jiwer torchaudio

In [ ]:
!git clone https://github.com/isi-nlp/uroman.git

In [ ]:
from huggingface_hub import login

login(token="your_hf_token_here")

In [ ]:
import json
import os
import shutil
import torch
import torchaudio
import soundfile as sf
import scipy.io.wavfile
from google.colab import files

# ==========================================
# 0. FIX VITS DEVANAGARI CRASH (UROMAN)
# ==========================================
uroman_path = "/content/uroman"
if not os.path.exists(uroman_path):
    print("Downloading uroman for VITS Devanagari support...")
    os.system(f"git clone https://github.com/isi-nlp/uroman.git {uroman_path}")

os.environ["UROMAN"] = uroman_path

from transformers import (
    VitsModel, AutoTokenizer,
    WhisperProcessor, WhisperForConditionalGeneration
)
from jiwer import wer, cer

# ==========================================
# 1. SETUP & DIRECTORIES
# ==========================================
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = "vits_maithili_outputs"
os.makedirs(output_dir, exist_ok=True)

with open('maithili_balanced_set.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

if isinstance(raw_data, dict):
    for key, value in raw_data.items():
        if isinstance(value, list):
            raw_data = value
            break

sentences = []
for item in raw_data:
    if isinstance(item, dict):
        text_keys = ['text', 'sentence', 'transcript', 'transcription', 'content']
        found = False
        for k in text_keys:
            if k in item:
                sentences.append(str(item[k]))
                found = True
                break
        if not found:
            for val in item.values():
                if isinstance(val, str):
                    sentences.append(val)
                    break
    elif isinstance(item, str):
        sentences.append(item)

print(f"Loaded {len(sentences)} sentences for evaluation.\n")
if len(sentences) > 0:
    print(f"Sample: {sentences[0]}\n")

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading VITS TTS Model (Meta MMS - Maithili)...")
vits_id = "facebook/mms-tts-mai"
vits_model = VitsModel.from_pretrained(vits_id).to(device)
vits_tokenizer = AutoTokenizer.from_pretrained(vits_id)

print("Loading Whisper Medium ASR Model...")
asr_id = "openai/whisper-medium"
asr_processor = WhisperProcessor.from_pretrained(asr_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_id).to(device)

forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def transcribe_audio_whisper(audio_path):
    """Loads audio, ensures 16kHz for Whisper, and transcribes."""
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    input_features = asr_processor(
        waveform.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    transcription = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

# ==========================================
# 4. GENERATION & EVALUATION LOOP
# ==========================================
results = []

for i, text in enumerate(sentences):
    print(f"Processing Sentence {i+1}/{len(sentences)}...")
    text = str(text)
    vits_path = os.path.join(output_dir, f"vits_sent_{i+1}.wav")

    # --- Generate VITS ---
    inputs = vits_tokenizer(text, return_tensors="pt").to(device)
    if inputs["input_ids"].shape[1] == 0:
        print(f"  -> WARNING: VITS Tokenizer failed to process sentence {i+1}. Skipping.")
        vits_transcription = ""
        vits_wer, vits_cer = 1.0, 1.0
    else:
        inputs["input_ids"] = inputs["input_ids"].long()
        with torch.no_grad():
            vits_out = vits_model(**inputs).waveform
        vits_audio = vits_out.squeeze().cpu().numpy()
        scipy.io.wavfile.write(vits_path, rate=vits_model.config.sampling_rate, data=vits_audio)

        vits_transcription = transcribe_audio_whisper(vits_path)
        vits_wer = wer(text, vits_transcription)
        vits_cer = cer(text, vits_transcription)

    results.append({
        "id": i + 1,
        "original_text": text,
        "transcription": vits_transcription,
        "wer": vits_wer,
        "cer": vits_cer
    })

# ==========================================
# 5. SAVE METRICS & ZIP ARCHIVE
# ==========================================
with open(os.path.join(output_dir, "evaluation_metrics.json"), 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("\nEvaluation Complete. Zipping files...")
zip_filename = "vits_maithili_audio"
shutil.make_archive(zip_filename, 'zip', output_dir)

print("Triggering download...")
files.download(f"{zip_filename}.zip")

